# Analisis Sentimen Sumbawa Hijau Lestari
## Analisis Sentimen & Penggalian Insight Tematik (dengan Kerangka ABSA & Analisis Keterlibatan)

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Setup Tampilan Plot
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10

# Path Direktori
PROCESSED_DIR = '../data/processed_dataset'
FIGURES_DIR = os.path.join(PROCESSED_DIR, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

## 1. Load & Validasi Data

In [ ]:
# Load Dataset Labelled yang dilengkapi metadata LikesCount
DATA_PATH = '../data/labelled/dataset_siap_label_sumbawa_hijau_lestari_labelled_with_like.xlsx'
df = pd.read_excel(DATA_PATH)

print(f"Dataset berhasil dimuat: {len(df)} baris, {len(df.columns)} kolom.")

### 1.1 Kerangka Aspect-Based Sentiment Analysis (ABSA)

Sentimen dalam dataset ini dinilai berdasarkan pendekatan **Aspect-Based Sentiment Analysis (ABSA)**, yaitu memisahkan objek penilaian menjadi dua aspek:
1. **Person**: Figur/akun pengunggah postingan (misalnya pejabat, instansi, atau warga).
2. **Program**: Substansi kebijakan "Sumbawa Hijau Lestari" itu sendiri.

Label Sentimen pada dataset ini secara konsisten mengacu pada **aspek Program**, bukan aspek Person. Konsekuensinya, komentar yang menunjukkan dukungan hangat kepada pengunggah postingan (misalnya ucapan semangat kepada pejabat atau warga yang menanam pohon) tetap diberi label **Netral** apabila komentar tersebut tidak menyatakan sikap eksplisit terhadap substansi program.

Berikut adalah tabel pemetaan kategori isu terhadap aspek yang dinilai:

| Kategori Isu | Aspek yang Dinilai |
| --- | --- |
| Obrolan/Balasan Personal | Person (pengunggah) |
| Apresiasi/Dukungan Program | Program |
| Kritik Kebijakan & Isu Terkait | Program |
| Pertanyaan/Klarifikasi Program | Program |
| Saran Teknis Pelaksanaan | Program |
| Kritik/Skeptisisme Keberlanjutan | Program |
| Informasi/Netral | Program |
| Spam/Promosi | Tidak relevan (dikeluarkan dari interpretasi aspek) |

> **Tegasan Kunci:** proporsi sentimen Netral yang tinggi pada dataset ini sebagian besar berasal dari volume komentar yang menyasar aspek Person, bukan indikasi sikap netral masyarakat terhadap Program.

## 2. Statistik Deskriptif & Visualisasi Distribusi

In [ ]:
sentimen_counts = df['Sentimen'].value_counts().reindex(['Positif', 'Netral', 'Negatif'])
sentimen_pct = (sentimen_counts / len(df)) * 100

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71', '#95a5a6', '#e74c3c']
bars = ax.bar(sentimen_counts.index, sentimen_counts.values, color=colors, edgecolor='black', linewidth=0.8)

ax.set_title('Distribusi Sentimen Publik Komentar Sumbawa Hijau Lestari', pad=15)
ax.set_xlabel('Sentimen')
ax.set_ylabel('Jumlah Komentar')
ax.set_ylim(0, max(sentimen_counts.values) * 1.15)

for bar, pct in zip(bars, sentimen_pct):
    yval = bar.get_height()
    ax.annotate(f'{yval} ({pct:.2f}%)',
                xy=(bar.get_x() + bar.get_width() / 2, yval),
                xytext=(0, 5),
                textcoords="offset points",
                ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, '01_distribusi_sentimen.png')
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved figure to {fig_path}")

In [ ]:
kat_counts = df['Kategori Isu'].value_counts()
kat_pct = (kat_counts / len(df)) * 100

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(kat_counts.index[::-1], kat_counts.values[::-1], color='#3498db', edgecolor='black', linewidth=0.8)

ax.set_title('Distribusi Kategori Isu Komentar Publik', pad=15)
ax.set_xlabel('Jumlah Komentar')
ax.set_ylabel('Kategori Isu')
ax.set_xlim(0, max(kat_counts.values) * 1.15)

for bar, pct in zip(bars, kat_pct[::-1]):
    width = bar.get_width()
    ax.annotate(f'{width} ({pct:.2f}%)',
                xy=(width, bar.get_y() + bar.get_height() / 2),
                xytext=(5, 0),
                textcoords="offset points",
                ha='left', va='center', fontweight='bold')

plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, '02_distribusi_kategori_isu.png')
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved figure to {fig_path}")

In [ ]:
ct_kat_sent = pd.crosstab(df['Kategori Isu'], df['Sentimen'])[['Positif', 'Netral', 'Negatif']]
ct_kat_sent.to_csv(os.path.join(PROCESSED_DIR, 'crosstab_kategori_sentimen.csv'))

fig, ax = plt.subplots(figsize=(10, 6))
ct_kat_sent.plot(kind='barh', stacked=True, color=['#2ecc71', '#95a5a6', '#e74c3c'], ax=ax, edgecolor='black', linewidth=0.5)

ax.set_title('Sebaran Sentimen per Kategori Isu', pad=15)
ax.set_xlabel('Jumlah Komentar')
ax.set_ylabel('Kategori Isu')
ax.legend(title='Sentimen')

plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, '03_crosstab_kategori_sentimen.png')
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved figure to {fig_path}")

In [ ]:
ct_jenis_sent = pd.crosstab(df['Jenis Komentar'], df['Sentimen'])[['Positif', 'Netral', 'Negatif']]
ct_jenis_sent.to_csv(os.path.join(PROCESSED_DIR, 'crosstab_jenis_komentar_sentimen.csv'))

fig, ax = plt.subplots(figsize=(8, 5))
ct_jenis_sent.plot(kind='bar', stacked=True, color=['#2ecc71', '#95a5a6', '#e74c3c'], ax=ax, edgecolor='black', linewidth=0.5)

ax.set_title('Proporsi Sentimen Berdasarkan Jenis Komentar (Utama vs Balasan)', pad=15)
ax.set_xlabel('Jenis Komentar')
ax.set_ylabel('Jumlah Komentar')
ax.legend(title='Sentimen')

plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, '04_jenis_komentar_sentimen.png')
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved figure to {fig_path}")

In [ ]:
top10_sumber = df['Nama Sumber'].value_counts().head(10).index
ct_sumber = pd.crosstab(df[df['Nama Sumber'].isin(top10_sumber)]['Nama Sumber'], df['Sentimen'])[['Positif', 'Netral', 'Negatif']]

fig, ax = plt.subplots(figsize=(10, 6))
ct_sumber.loc[top10_sumber[::-1]].plot(kind='barh', stacked=True, color=['#2ecc71', '#95a5a6', '#e74c3c'], ax=ax, edgecolor='black', linewidth=0.5)

ax.set_title('Distribusi Sentimen pada 10 Sumber Postingan Terbanyak', pad=15)
ax.set_xlabel('Jumlah Komentar')
ax.set_ylabel('Nama Sumber')
ax.legend(title='Sentimen')

plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, '05_top10_nama_sumber_sentimen.png')
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved figure to {fig_path}")

In [ ]:
df['Tanggal_dt'] = pd.to_datetime(df['Tanggal'])
df['Bulan_Tahun'] = df['Tanggal_dt'].dt.to_period('M')

tren_sentimen = pd.crosstab(df['Bulan_Tahun'], df['Sentimen'])[['Positif', 'Netral', 'Negatif']]

fig, ax = plt.subplots(figsize=(10, 5))
tren_sentimen.plot(kind='line', marker='o', color=['#2ecc71', '#95a5a6', '#e74c3c'], ax=ax, linewidth=2)

ax.set_title('Tren Volume Sentimen Publik per Bulan', pad=15)
ax.set_xlabel('Bulan')
ax.set_ylabel('Jumlah Komentar')
ax.legend(title='Sentimen')

plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, '06_tren_sentimen_bulanan.png')
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved figure to {fig_path}")

## 3. Penggalian Insight Tematik (Kualitatif + Kuantitatif)

In [ ]:
df_kritis = df[df['Kategori Isu'].isin(['Kritik/Skeptisisme Keberlanjutan', 'Kritik Kebijakan & Isu Terkait'])]
print(f"Jumlah Komentar Kritis: {len(df_kritis)}")

print("\nContoh Komentar Kritis:")
for i, (_, row) in enumerate(df_kritis.head(5).iterrows(), 1):
    print(f"{i}. [{row['Kategori Isu']}] ({row['Nama Sumber']}): \"{row['Isi Komentar']}\"")

In [ ]:
kw_pohon = ['lokal', 'spesies', 'jenis', 'sengon', 'beringin', 'asam', 'bage', 'jagung', 'hutan', 'pohon', 'tanam']

kw_counts = {}
for kw in kw_pohon:
    count = df_kritis['Isi Komentar'].str.contains(kw, case=False, na=False).sum()
    kw_counts[kw] = count

print("Frekuensi Kemunculan Kata Kunci Isu Kritis pada Komentar Kritis:")
for kw, c in sorted(kw_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"- '{kw}': {c} komentar")

In [ ]:
df_apresiasi = df[df['Kategori Isu'] == 'Apresiasi/Dukungan Program']
print(f"Jumlah Komentar Apresiasi Program: {len(df_apresiasi)}")

print("\nContoh Komentar Apresiasi Program:")
for i, (_, row) in enumerate(df_apresiasi.head(4).iterrows(), 1):
    print(f"{i}. [{row['Nama Sumber']}]: \"{row['Isi Komentar']}\"")

## 4. Analisis Kata Kunci & Visualisasi Wordcloud (Kategori Kritis)

In [ ]:
INDONESIAN_STOPWORDS = set([
    'yang', 'di', 'ke', 'dari', 'dan', 'ini', 'itu', 'dengan', 'untuk', 'pada', 'adalah',
    'sebagai', 'akan', 'bisa', 'juga', 'ada', 'tidak', 'tapi', 'ya', 'gak', 'kalo', 'kalau',
    'sama', 'lagi', 'sudah', 'sdh', 'tdk', 'yg', 'utk', 'dngn', 'dgn', 'oleh', 'hanya',
    'atau', 'apakah', 'ia', 'kita', 'kami', 'mereka', 'saya', 'aku', 'anda', 'bapak', 'pak',
    'saja', 'bisa', 'harus', 'masih', 'banyak', 'kurang', 'apa', 'siapa', 'mengapa', 'bagaimana'
])

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    tokens = [w for w in text.split() if len(w) > 2 and w not in INDONESIAN_STOPWORDS]
    return tokens

texts_kritis = df_kritis['Isi Komentar'].apply(preprocess_text)
all_words_kritis = [word for tokens in texts_kritis for word in tokens]

word_freq = pd.Series(all_words_kritis).value_counts().head(20)
df_word_freq = pd.DataFrame({'Kata': word_freq.index, 'Frekuensi': word_freq.values})
df_word_freq.to_csv(os.path.join(PROCESSED_DIR, 'top_words_kritis.csv'), index=False)

# Generate Wordcloud
clean_text_joined = ' '.join(all_words_kritis)
wc = WordCloud(width=800, height=500, background_color='white', colormap='Reds', max_words=50, random_state=42).generate(clean_text_joined)
wc_path = os.path.join(FIGURES_DIR, '09_wordcloud_kategori_kritis.png')
wc.to_file(wc_path)
print(f"Saved wordcloud figure to {wc_path}")

# Visualisasi Berdampingan (Bar Chart Frekuensi Kata & Wordcloud)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Subplot 1: Bar Chart Horizontal
bars = ax1.barh(df_word_freq['Kata'][::-1], df_word_freq['Frekuensi'][::-1], color='#e74c3c', edgecolor='black', linewidth=0.8)
ax1.set_title('Top 20 Kata Paling Sering Muncul di Komentar Kritis', fontsize=12, fontweight='bold', pad=10)
ax1.set_xlabel('Frekuensi')
ax1.set_ylabel('Kata')
ax1.set_xlim(0, max(df_word_freq['Frekuensi']) * 1.15)
for bar, count in zip(bars, df_word_freq['Frekuensi'][::-1]):
    ax1.annotate(str(count), xy=(bar.get_width(), bar.get_y() + bar.get_height() / 2), xytext=(5, 0),
                 textcoords="offset points", ha='left', va='center', fontweight='bold')

# Subplot 2: Wordcloud
ax2.imshow(wc, interpolation='bilinear')
ax2.axis('off')
ax2.set_title('Wordcloud Kata Kunci Komentar Kritis', fontsize=12, fontweight='bold', pad=10)

plt.tight_layout()
fig_bar_path = os.path.join(FIGURES_DIR, '07_top_words_kategori_kritis.png')
plt.savefig(fig_bar_path, dpi=300)
plt.show()
print(f"Saved combined figures to {fig_bar_path} and {wc_path}")

## 5. Analisis Keterlibatan (Engagement) Berbasis Jumlah Komentar dan Like

In [ ]:
# Load data mentah & daftar link sumber
RAW_DATA_PATH = '../data/dataset/dataset_analisis_sumbawa_hijau_lestari.csv'
LINKS_PATH = '../data/dataset/daftar_link_sumber.csv'

df_raw_eng = pd.read_csv(RAW_DATA_PATH)
df_links_eng = pd.read_csv(LINKS_PATH)

total_posts = 54
total_raw_comments = len(df_raw_eng)
avg_raw_per_post = total_raw_comments / total_posts

total_clean_comments = len(df)
avg_clean_per_post = total_clean_comments / total_posts

avg_likes_per_comment = df['Jumlah Like'].mean()
total_likes = df['Jumlah Like'].sum()
avg_likes_per_post = total_likes / total_posts

print(f"=== RINGKASAN METRIK KETERLIBATAN (ENGAGEMENT) ===")
print(f"- Total Komentar Mentah (Scraped)   : {total_raw_comments} (Rata-rata: {avg_raw_per_post:.2f} per postingan)")
print(f"- Total Komentar Bersih (Dianalisis): {total_clean_comments} (Rata-rata: {avg_clean_per_post:.2f} per postingan)")
print(f"- Rata-rata Like per Komentar       : {avg_likes_per_comment:.3f} likes/komentar")
print(f"- Rata-rata Like per Postingan      : {avg_likes_per_post:.2f} likes/postingan")

# Hitung distribusi per postingan (54 postingan)
clean_counts_per_link = df.groupby('Link Sumber').size()
likes_per_link = df.groupby('Link Sumber')['Jumlah Like'].sum()

df_engagement_post = df_links_eng.copy()
df_engagement_post['Jumlah Komentar Bersih'] = df_engagement_post['Link'].map(clean_counts_per_link).fillna(0).astype(int)
df_engagement_post['Total Like Komentar'] = df_engagement_post['Link'].map(likes_per_link).fillna(0).astype(int)
df_engagement_post = df_engagement_post.sort_values(by='Jumlah Komentar Bersih', ascending=False).reset_index(drop=True)

# Export ringkasan ke CSV
csv_eng_path = os.path.join(PROCESSED_DIR, 'ringkasan_keterlibatan_postingan.csv')
df_engagement_post.to_csv(csv_eng_path, index=False)
print(f"Saved engagement summary CSV to: {csv_eng_path}")

# Tampilkan tabel ringkas postingan tertinggi & terendah
top_bottom_summary = pd.concat([
    df_engagement_post.head(3),
    df_engagement_post.tail(3)
])[['Nama Sumber', 'Link', 'Jumlah Komentar Bersih', 'Total Like Komentar']]

print("\nTabel Ringkas Postingan Komentar Terbanyak & Tersedikit:")
print(top_bottom_summary.to_string(index=False))

# Plot Bar Horizontal Jumlah Komentar per Postingan (54 Postingan)
fig, ax = plt.subplots(figsize=(10, 14))
labels = [f"{row['Nama Sumber']} (Post #{row['No']})" for _, row in df_engagement_post.iterrows()]
bars = ax.barh(labels[::-1], df_engagement_post['Jumlah Komentar Bersih'][::-1], color='#2980b9', edgecolor='black', linewidth=0.5)

ax.set_title('Sebaran Jumlah Komentar Bersih per Postingan (54 Postingan Facebook)', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Jumlah Komentar Bersih')
ax.set_ylabel('Akun / Sumber Postingan')
ax.set_xlim(0, max(df_engagement_post['Jumlah Komentar Bersih']) * 1.15)

for bar in bars:
    width = bar.get_width()
    if width > 0:
        ax.annotate(str(width),
                    xy=(width, bar.get_y() + bar.get_height() / 2),
                    xytext=(4, 0),
                    textcoords="offset points",
                    ha='left', va='center', fontsize=8, fontweight='bold')

plt.tight_layout()
fig_eng_path = os.path.join(FIGURES_DIR, '08_keterlibatan_per_postingan.png')
plt.savefig(fig_eng_path, dpi=300)
plt.show()
print(f"Saved engagement figure to: {fig_eng_path}")

## 6. Kesimpulan & Rekomendasi

### 6.1 Kesimpulan Berbasis Data

1. **Pemisahan Aspek & Asal Sentimen Netral**:
   Setelah dipisahkan berdasarkan kerangka Aspect-Based Sentiment Analysis (ABSA), proporsi sentimen Netral yang tinggi pada dataset ini (56.11%, 225 komentar) sebagian besar berasal dari komentar yang menyasar aspek **Person** (pengunggah/figur), bukan aspek **Program**. Kategori `Obrolan/Balasan Personal` mencakup **231 dari 401 komentar (57.61%)** sebagai dasar kuantitatif utama temuan ini.

2. **Perbandingan Respon Eksplisit Aspek Program**:
   Di antara komentar yang benar-benar menyasar aspek Program secara eksplisit, kelompok `Apresiasi/Dukungan Program` (**63 komentar**, seluruhnya bersentimen Positif) lebih banyak dibandingkan gabungan dua kategori kritik terhadap Program, yaitu `Kritik Kebijakan & Isu Terkait` (29 komentar) dan `Kritik/Skeptisisme Keberlanjutan` (17 komentar) yang berjumlah **46 komentar** (seluruhnya bersentimen Negatif). Perbandingan eksplisit ini mencatatkan angka **63 berbanding 46**.

3. **Tingkat Keterlibatan (Engagement) Publik**:
   Tingkat keterlibatan warga terhadap program di media sosial tergolong rendah, berdasarkan tiga indikator empiris:
   - Rata-rata komentar bersih per postingan hanya sekitar **7 hingga 8 komentar** (7.43 komentar/postingan dari total 54 postingan).
   - Jumlah komentar tertinggi pada satu postingan hanya mencapai **22 komentar**.
   - Mayoritas komentar yang masuk (57.61%) justru menyasar aspek Person (sapaan/obrolan personal), bukan aspek Program.

----

### 6.2 Catatan Interpretatif Peneliti

1. **Atensi Publik terhadap Program**:
   Rendahnya volume komentar yang secara eksplisit membahas Program, dikombinasikan dengan dominasi komentar bernuansa dukungan personal ke pengunggah, mengindikasikan bahwa program "Sumbawa Hijau Lestari" belum berhasil menarik perhatian publik secara luas di media sosial, terlepas dari sentimen yang ada.

2. **Dua Sudut Pandang Dominasi Apresiasi**:
   Di antara komentar yang secara eksplisit menyasar Program, kecenderungan apresiasi yang lebih tinggi dibanding kritik (63 berbanding 46) dapat dibaca dari dua arah yang berbeda:
   - **Perspektif Pertama**: Sebagai sikap netral masyarakat karena minimnya komentar spesifik yang membahas program secara mendalam.
   - **Perspektif Kedua**: Sebagai kecenderungan psikologis umum terhadap program pemerintah, yaitu warga cenderung tidak banyak memberikan tanggapan eksplisit terhadap program yang tidak mereka anggap bermasalah, sehingga rendahnya penolakan eksplisit tidak serta-merta berarti persetujuan aktif.

3. **Status Hipotesis & Arah Penelitian Lanjutan**:
   Peneliti menegaskan bahwa asumsi "minimnya tanggapan negatif berarti persetujuan diam-diam terhadap program" adalah **hipotesis interpretatif**, bukan kesimpulan yang telah teruji secara statistik. Untuk memvalidasi hipotesis ini diperlukan pengujian statistik lanjutan (misalnya uji-t atau uji-z untuk membandingkan proporsi sentimen antar kelompok, atau metode inferensia lain yang relevan), yang berada di luar cakupan analisis deskriptif pada notebook ini dan menjadi arah bagi penelitian lanjutan.

----

### 6.3 Rumusan Rekomendasi Kebijakan Program

| No | Rekomendasi | Rasional Kebijakan |
| --- | --- | --- |
| 1 | **Mekanisme Evaluasi & Pemeliharaan Berkala Pasca-Penanaman** | Memastikan ketersediaan anggaran dan petugas pemeliharaan agar bibit yang ditanam dapat tumbuh berkelanjutan, bukan sekadar kegiatan seremoni penanaman awal. |
| 2 | **Pemilihan Bibit Pohon Lokal & Edukasi Publik** | Memprioritaskan bibit tanaman lokal berakar kuat dan bermanfaat ekologis/ekonomis panjang, seperti asam jawa (*bage*) dan beringin, disertai penyampaian tata cara perawatan kepada masyarakat. |
| 3 | **Penyelarasan Kebijakan Penghijauan dengan Penataan Lahan** | Memperketat regulasi pembukaan hutan untuk ladang jagung di area lereng dan kawasan tangkapan air, agar kegiatan penghijauan tidak kontradiktif dengan alih fungsi hutan yang masih berlangsung. |